In [ ]:
import os
from typing import Optional

try:
    from dotenv import load_dotenv
    load_dotenv()
except ModuleNotFoundError:
    # Fallback if python-dotenv isn't installed; OS env vars still work
    pass

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://fantasy.sixnationsrugby.com"


def _build_session() -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=3, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504]
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    return s


def fetch_match(
    match_id: int,
    language: str = "en",
    token: Optional[str] = None,
    x_access_key: Optional[str] = None,
) -> dict:
    """Fetch a match payload from the Six Nations Fantasy API.

    Provide your browser token via the SIXNATIONS_TOKEN env var (without or with the leading 'Token ').
    Optionally override x-access-key via SIXNATIONS_X_ACCESS_KEY.
    """
    url = f"{BASE_URL}/v1/private/match/{match_id}?lg={language}"
    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "cache-control": "no-cache",
        "pragma": "no-cache",
    }
    if x_access_key is None:
        x_access_key = os.getenv("SIXNATIONS_X_ACCESS_KEY", "600@18.23@")
    headers["x-access-key"] = x_access_key

    if token is None:
        token = os.getenv("SIXNATIONS_TOKEN")
    if token:
        headers["authorization"] = (
            token if token.lower().startswith("token ") else f"Token {token}"
        )

    sess = _build_session()
    r = sess.get(url, headers=headers, allow_redirects=True, timeout=20)
    if r.status_code in (401, 403):
        raise RuntimeError(
            "Unauthorized (401/403). Set SIXNATIONS_TOKEN env var to your Authorization token while logged in to fantasy.sixnationsrugby.com."
        )
    r.raise_for_status()
    return r.json()


MATCH_ID = int(os.getenv("SIXNATIONS_MATCH_ID", "1"))
try:
    data = fetch_match(MATCH_ID)
    print(f"Fetched match {MATCH_ID}")
except Exception as e:
    print("Error fetching match data:", e)
    data = None


ModuleNotFoundError: No module named 'dotenv'

In [ ]:
if data is None:
    raise SystemExit(
        "Set SIXNATIONS_TOKEN environment variable and re-run. Log in to fantasy.sixnationsrugby.com, open DevTools -> Network, copy the Authorization header from an API call."
    )


In [ ]:
import pandas as pd
from IPython.display import display

# -------------------------------------------------------
# POSITION + FORWARD/BACK MAPPING
# -------------------------------------------------------
position_map = {
    "6": "Back-three",
    "7": "Centre",
    "8": "Fly-half",
    "9": "Scrum-half",
    "10": "Back-row",
    "11": "Second-row",
    "12": "Prop",
    "13": "Hooker",
}

forwards = {"Back-row", "Second-row", "Prop", "Hooker"}
backs = {"Back-three", "Centre", "Fly-half", "Scrum-half"}

# -------------------------------------------------------
# SCORING RULES
# -------------------------------------------------------
scoring = {
    "Try": None,
    "Assists": 4,
    "Conversion": 2,
    "Penalty": 3,
    "DropGoal": 5,
    "DefendersBeaten": 2,
    "MetresCarried": 0.1,
    "FiftyTwentyTwo": 7,
    "KicksRecovered": 2,
    "Offloads": 2,
    "AttackingScrumWin": 1,
    "Tackles": 1,
    "BreakdownSteal": 5,
    "LineoutSteal": 7,
    "PenaltyConceded": -1,
    "PlayerOfTheMatch": 15,
    "YellowCard": -5,
    "RedCard": -8,
    "Minutes": 0,
}

# -------------------------------------------------------
# FRIENDLY COLUMN NAMES
# -------------------------------------------------------
stat_name_map = {
    "Min": "Minutes",
    "T": "Try",
    "As": "Assists",
    "C": "Conversion",
    "Pen": "Penalty",
    "MC": "MetresCarried",
    "DB": "DefendersBeaten",
    "Ta": "Tackles",
    "CPen": "PenaltyConceded",
    "50-22": "FiftyTwentyTwo",
    "KR": "KicksRecovered",
    "DG": "DropGoal",
    "OF": "Offloads",
    "LS": "LineoutSteal",
    "BS": "BreakdownSteal",
    "POTM": "PlayerOfTheMatch",
    "SW": "AttackingScrumWin",
    "YC": "YellowCard",
    "RC": "RedCard",
}


def _to_int_or_zero(v) -> int:
    try:
        if v is None:
            return 0
        if isinstance(v, int):
            return int(v)
        s = str(v).strip().replace(",", "")
        return int(float(s))
    except Exception:
        return 0


def _to_float_or_zero(v) -> float:
    try:
        if v is None:
            return 0.0
        if isinstance(v, (int, float)):
            return float(v)
        s = str(v).strip().replace(",", "")
        return float(s)
    except Exception:
        return 0.0


def flatten_players(players, team_name, legend):
    rows = []
    legend_map = {}
    for i, lg in enumerate(legend):
        short = lg.get("label_short", "")
        legend_map[i] = stat_name_map.get(short, f"crit_{i}")

    for p in players:
        pos_key = str(p.get("position", ""))
        pos_name = position_map.get(pos_key, pos_key)
        row = {
            "id": p["id"],
            "name": p["nom"],
            "position": pos_name,
            "points_total": p["points"],
            "team": team_name,
        }
        for i, crit in enumerate(p.get("criteres", [])):
            stat_col = legend_map.get(i, f"crit_{i}")
            raw_val = crit.get("value")
            val_num = (
                _to_float_or_zero(raw_val)
                if stat_col == "MetresCarried"
                else _to_int_or_zero(raw_val)
            )
            row[stat_col] = val_num
            if stat_col == "Try":
                row["Try_points"] = (10 if pos_name in backs else 15) * int(val_num)
            elif stat_col == "MetresCarried":
                # Scoring is 1 point per full 10 metres carried (floor), e.g. 19 -> 1, 20 -> 2
                # Use floor division to avoid floating point rounding artifacts.
                row["MetresCarried_points"] = int(val_num // 10)
            else:
                if stat_col in scoring:
                    row[f"{stat_col}_points"] = int(val_num) * scoring[stat_col]
        rows.append(row)
    return pd.DataFrame(rows)


legend = data["match"]["legende"]
df_dom = flatten_players(data["match"]["joueursdom"], "France", legend)
df_ext = flatten_players(data["match"]["joueursext"], "Ireland", legend)
df = pd.concat([df_dom, df_ext], ignore_index=True)

base_cols = ["id", "name", "team", "position", "points_total"]
ordered_stats = [
    "Minutes",
    "Try",
    "Assists",
    "Conversion",
    "Penalty",
    "MetresCarried",
    "DefendersBeaten",
    "Tackles",
    "PenaltyConceded",
    "FiftyTwentyTwo",
    "KicksRecovered",
    "DropGoal",
    "Offloads",
    "LineoutSteal",
    "BreakdownSteal",
    "PlayerOfTheMatch",
    "AttackingScrumWin",
    "YellowCard",
    "RedCard",
]
ordered_cols = []
for s in ordered_stats:
    if s in df.columns:
        ordered_cols.append(s)
    pts_col = f"{s}_points"
    if pts_col in df.columns:
        ordered_cols.append(pts_col)
extra_cols = [c for c in df.columns if c not in base_cols + ordered_cols]
df = df[base_cols + ordered_cols + extra_cols]

numeric_cols = [c for c in df.columns if c not in {"id", "name", "team", "position"}]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

point_cols = [c for c in df.columns if c.endswith("_points")]
df["computed_points_total"] = df[point_cols].sum(axis=1)
df["points_delta"] = df["points_total"] - df["computed_points_total"]

mismatches = df[df["points_delta"] != 0]
display(df)
display(mismatches)


In [ ]:
# Persist data for dashboarding
import os
from datetime import datetime
os.makedirs('data', exist_ok=True)
ts = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
base = f'data/match_{MATCH_ID}_{ts}'
df.to_parquet(base + '.parquet', index=False)
df.to_csv(base + '.csv', index=False)
print('Saved:', base + '.parquet', 'and', base + '.csv')
